In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [2]:
disprot = pd.read_csv("../data/disprot.tsv", sep='\t', low_memory=False)
disprot = disprot.rename(columns={
    'UniProt ACC':'acc', 'Organism':'organism',
    'Term namespace':'term_namespace', 'Start':'start', 'End':'end',
    'Term name':'term_name',
})
disprot['acc'] = disprot['acc'].str.split('-').str[0]

# Structural state annotations = disordered regions
state = disprot[(disprot['organism']=='Homo sapiens') &
                (disprot['term_namespace']=='Structural state')].copy()
print(f"Disorder annotations (Structural state, human): {len(state)}")
print(f"Distinct proteins with disorder annotations: {state['acc'].nunique()}")

# Build per-acc list of (start, end) tuples (1-indexed, inclusive)
disorder_regions = (state.groupby('acc')
                         .apply(lambda x: [(int(s), int(e)) for s, e in zip(x['start'], x['end'])])
                         .to_dict())

# Sanity: distribution of disordered residues per protein
n_dis = {acc: sum(e - s + 1 for s, e in regs)
         for acc, regs in disorder_regions.items()}
print(f"\nDisordered residues per protein:")
print(pd.Series(n_dis).describe().round(1))

Disorder annotations (Structural state, human): 3231
Distinct proteins with disorder annotations: 1278

Disordered residues per protein:
count     1278.0
mean       214.3
std        635.3
min         10.0
25%         28.0
50%         64.0
75%        171.8
max      16344.0
dtype: float64


In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [7]:
import torch, esm
from tqdm.auto import tqdm

device = "mps" if torch.backends.mps.is_available() else \
         ("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

print("Loading ESM-2 650M (cached from W5) ...")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()
model = model.to(device)

seqs = pd.read_csv("../data/sequences.csv")
MAX_LEN = 1022

@torch.no_grad()
def embed_disorder(seq, acc):
    """Mean-pool ESM-2 token reps over DisProt disordered residues only.
       Returns (embedding, fallback_flag). fallback_flag=True means we
       fell back to whole-protein pooling because no disorder residues
       were in the truncation window."""
    s = seq[:MAX_LEN]
    _, _, toks = batch_converter([("p", s)])
    toks = toks.to(device)
    out = model(toks, repr_layers=[33])
    # representations[33] is (1, L+2, 1280); strip CLS at idx 0 and EOS at idx -1
    per_residue = out["representations"][33][0, 1:-1]   # shape (L, 1280)
    L = per_residue.shape[0]

    regions = disorder_regions.get(acc, [])
    if not regions:
        return per_residue.mean(dim=0).cpu().numpy().astype("float32"), True

    # Build residue indices (0-indexed) within truncation window
    indices = []
    for start, end in regions:
        for i in range(start - 1, min(end, L)):  # 1-indexed -> 0-indexed
            indices.append(i)

    if not indices:
        return per_residue.mean(dim=0).cpu().numpy().astype("float32"), True

    idx_t = torch.tensor(indices, device=device, dtype=torch.long)
    selected = per_residue[idx_t]
    return selected.mean(dim=0).cpu().numpy().astype("float32"), False

embeddings = {}
fallbacks = []
for _, row in tqdm(seqs.iterrows(), total=len(seqs)):
    try:
        emb, fb = embed_disorder(row['sequence'], row['acc'])
        embeddings[row['acc']] = emb
        if fb:
            fallbacks.append(row['acc'])
    except Exception as e:
        print(f"  Failed for {row['acc']}: {type(e).__name__}: {e}")

accs = list(embeddings.keys())
X = np.stack([embeddings[a] for a in accs])
np.savez_compressed("../data/features_esm2_disorder.npz",
                    accs=np.array(accs), X=X.astype("float32"))
print(f"\nSaved {X.shape[0]} disorder-pooled embeddings of dim {X.shape[1]}")
print(f"Fallback to whole-protein pooling (no disorder in truncation window): "
      f"{len(fallbacks)} proteins")

Using device: mps
Loading ESM-2 650M (cached from W5) ...


  0%|          | 0/1279 [00:00<?, ?it/s]


Saved 1279 disorder-pooled embeddings of dim 1280
Fallback to whole-protein pooling (no disorder in truncation window): 41 proteins


In [8]:
data = np.load("../data/features_esm2_disorder.npz", allow_pickle=True)
X, accs = data['X'], data['accs']
norms = np.linalg.norm(X, axis=1)
print("shape:", X.shape, "dtype:", X.dtype)
print(f"L2 norms: min={norms.min():.2f}, max={norms.max():.2f}, mean={norms.mean():.2f}")
print(f"NaNs? {np.isnan(X).any()}; zero-norm vectors? {(norms == 0).sum()}")

data_old = np.load("../data/features_esm2.npz", allow_pickle=True)
X_old = data_old['X']
accs_old = list(data_old['accs'])
idx_old = {a: i for i, a in enumerate(accs_old)}
common = [a for a in accs if a in idx_old]
X_new_c = np.stack([X[list(accs).index(a)] for a in common])
X_old_c = np.stack([X_old[idx_old[a]] for a in common])

def cosine(a, b):
    return (a * b).sum(axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))

cs = cosine(X_new_c, X_old_c)
print(f"\nCosine similarity between disorder-pooled and whole-protein embeddings:")
print(f"  median={np.median(cs):.3f}, mean={cs.mean():.3f}, "
      f"min={cs.min():.3f}, max={cs.max():.3f}")

Task was destroyed but it is pending!
task: <Task pending name='Task-439' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/miniconda3/envs/biol466/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-440' coro=<Kernel.shell_main() running at /opt/miniconda3/envs/biol466/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/miniconda3/envs/biol466/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/miniconda3/envs/biol466/lib/python3.11/site-packages/IPython/core/compilerop.py:86: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
Task was destroyed but it is pending!
task: <Task pending name='Task-440' coro=<Kernel.shell_main() running at /opt/miniconda3/envs/biol466/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup(

shape: (1279, 1280) dtype: float32
L2 norms: min=4.78, max=10.18, mean=8.75
NaNs? False; zero-norm vectors? 0

Cosine similarity between disorder-pooled and whole-protein embeddings:
  median=0.946, mean=0.927, min=0.475, max=1.000
